# 🧠 Fetalytics — Cloud AI Training on Free GPU (Google Colab)
Train the **Fetalytics FetalHeadUNet-v2.0** ultrasound segmentation and standard-plane classification model in the cloud for **free** using Google Colab's Nvidia T4 GPU (16 GB VRAM).

### Why train in the cloud instead of your laptop?
- ⚡ **Blazing Fast**: Training 340 ultrasound cases takes **~2 to 3 minutes** on an Nvidia T4 GPU vs ~45-60+ minutes on a laptop CPU.
- ❄️ **Zero Laptop Strain**: Zero CPU heating, zero battery drain, zero fan noise.
- 📁 **Handles Hundreds of Videos & Files Effortlessly**: The pipeline reads keyframes directly from video files without overloading RAM.
- ☁️ **Automated Checkpoints**: Saves the trained `.pt` model back to Google Drive, Backblaze B2, and Neon PostgreSQL.

---
### Quick Setup (3 Steps):
1. Ensure GPU is enabled: **Runtime** > **Change runtime type** > **T4 GPU** (Free).
2. Choose how to load your Google Drive dataset (**Option A**: Mount Drive, or **Option B**: Direct `gdown`).
3. Click **Runtime** > **Run all** (or press `Ctrl + F9`).

### Step 1: Verify Free Nvidia GPU in Colab
Let's verify that Google Colab has assigned a free Nvidia GPU (T4 or better).

In [ ]:
!nvidia-smi
import torch

print("\n" + "=" * 65)
if torch.cuda.is_available():
    print(f"✅ Hardware GPU Active: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM Available:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"✅ PyTorch Version:     {torch.__version__}")
else:
    print("⚠️ No GPU detected! Please go to: Runtime > Change runtime type > Select T4 GPU")
print("=" * 65)

### Step 2: Clone Repository & Install Dependencies
Clones the latest Fetalytics codebase directly from your GitHub repository into Colab.

In [ ]:
# Clone the latest Fetalytics repository from GitHub
%cd /content
!rm -rf /content/Fetalytics
!git clone https://github.com/mohamedrayenlaimich/Fetalytics.git
%cd /content/Fetalytics

# Install required dependencies
!pip install -q boto3 botocore sqlalchemy psycopg2-binary pydantic python-dotenv gdown
print("\n✅ Repository cloned and dependencies installed successfully!")

### Step 3: Connect Your Google Drive Dataset
Your Google Drive folder:  
🔗 `https://drive.google.com/drive/u/3/folders/1am61JzoUFojmqvhQ1ztc2dllkmxtmlSD` (Folder ID: `1am61JzoUFojmqvhQ1ztc2dllkmxtmlSD`)

Choose **Option A** (recommended: mount your Google Drive) OR **Option B** (direct download with `gdown`).

In [ ]:
# OPTION A: Mount your Google Drive (Recommended)
# If the folder is shared with your Google account, you can right-click it in Google Drive
# and select 'Add shortcut to Drive' -> 'My Drive'. Then it will appear at /content/drive/MyDrive/...

from google.colab import drive
drive.mount('/content/drive')

import os, glob
print("\nChecking mounted Google Drive contents:")
my_drive = "/content/drive/MyDrive"
found_datasets = glob.glob(f"{my_drive}/*dataset*", recursive=False) + glob.glob(f"{my_drive}/*Dataset*", recursive=False) + glob.glob(f"{my_drive}/*fetal*", recursive=False)
for d in found_datasets:
    print("  Found:", d)

In [ ]:
# OPTION B (Alternative): Download folder directly using gdown
# Folder ID: 1am61JzoUFojmqvhQ1ztc2dllkmxtmlSD

FOLDER_ID = "1am61JzoUFojmqvhQ1ztc2dllkmxtmlSD"
OUT_DIR = "/content/dataset_drive"

# If the folder has link sharing enabled ('Anyone with link can view'), gdown downloads it directly:
# !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O {OUT_DIR}

print("Drive dataset setup configured.")

### Step 3b: Inspect Dataset Files (Videos, Masks & Metadata)
Let's inspect the files in your dataset (ultrasound video clips `.avi`/`.mp4`, ground truth masks `.png`, and annotation `.csv`/`.json` files).

In [ ]:
from pathlib import Path
from collections import Counter

# Look for dataset files in Drive or local temporary directories
search_dirs = [
    Path("/content/dataset_drive"),
    Path("/content/drive/MyDrive"),
    Path("/content/Fetalytics/tmp_ingest/DatasetV3/extracted"),
    Path("tmp_ingest/DatasetV3/extracted"),
]

found_dir = None
for d in search_dirs:
    if d.exists() and (list(d.rglob("*.avi")) or list(d.rglob("*.mp4")) or list(d.rglob("*.zip"))):
        found_dir = d
        break

if found_dir:
    all_files = [f for f in found_dir.rglob("*") if f.is_file()]
    ext_counts = Counter(f.suffix.lower() for f in all_files)
    vids = list(found_dir.rglob("*.avi")) + list(found_dir.rglob("*.mp4"))
    masks = list(found_dir.rglob("*.png"))
    
    print("=" * 65)
    print(f"📊 DATASET INVENTORY BREAKDOWN ({found_dir})")
    print("=" * 65)
    print(f"🎥 Ultrasound Videos (.avi/.mp4): {len(vids)}")
    print(f"🎭 Segmentation Masks (.png):      {len(masks)}")
    print("📁 All file types found:")
    for ext, cnt in ext_counts.most_common():
        print(f"   {ext or 'NO EXT'}: {cnt} files")
    print("=" * 65)
    print("ℹ️ Note: Fetalytics does NOT load all raw videos into RAM at once.")
    print("   It intelligently seeks the clinically annotated standard-plane keyframes,")
    print("   pairs each with its corresponding segmentation mask, and trains with zero RAM overflow!")
else:
    print("⚠️ No dataset folder found yet. Run Step 3 (Option A or Option B) first, or dataset will stream from Backblaze B2.")

### Step 4: Configure Cloud Credentials (Backblaze B2 & Neon PostgreSQL)
These credentials allow saving the trained model checkpoint to Backblaze B2 and logging the training run to Neon PostgreSQL.

In [ ]:
import os

# Backblaze B2 S3-Compatible Storage Configuration
os.environ["B2_KEY_ID"] = "0055b5fe7a3435b0000000003"
os.environ["B2_APPLICATION_KEY"] = "K005GfdVPN3V8DM+3MV8D8ZrSGHIvkM"
os.environ["B2_ENDPOINT_URL"] = "s3.us-east-005.backblazeb2.com"
os.environ["B2_BUCKET_NAME"] = "fetalytics-videos"
os.environ["B2_REGION_NAME"] = "us-east-005"

# Neon PostgreSQL Connection String (Loaded from environment or paste your URL)
os.environ["DATABASE_URL"] = os.environ.get("DATABASE_URL", "")

print("✅ Cloud credentials configured successfully!")

### Step 5: Run Fast Cloud Training on Nvidia T4 GPU
This single command:
1. Loads the dataset from your Google Drive folder (or Backblaze B2).
2. Partitions by subject (100 train, 21 val, 22 test) with zero patient leakage.
3. Trains `FetalHeadUNet-v2.0` on the Nvidia T4 GPU in **~2 to 3 minutes**.
4. Evaluates strictly on the held-out test split.
5. Uploads the trained `.pt` checkpoint to Backblaze B2 and saves a copy to Google Drive.
6. Registers the run in Neon PostgreSQL.

In [ ]:
# Make sure we are inside the Fetalytics directory
%cd /content/Fetalytics

# Run training (auto-detects Drive dataset or Backblaze B2)
!python AI/training/train_cloud.py --epochs 12 --batch-size 16 --lr 0.003 --seed 42 --patience 4

### Step 6: Visualize Held-Out Test Evaluation & Diagnostic Sheets
Inspect the final evaluation metrics and side-by-side diagnostic image overlays.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

report_p = Path("/content/Fetalytics/results/reports/training_evaluation_report.json")
if not report_p.exists():
    report_p = Path("results/reports/training_evaluation_report.json")

if report_p.exists():
    with open(report_p, "r", encoding="utf-8") as f:
        rep = json.load(f)
    
    seg = rep["evaluation_metrics"]["segmentation"]
    cls_m = rep["evaluation_metrics"]["standard_plane_classification"]
    bio = rep["evaluation_metrics"]["biometry_and_landmarks"]
    
    print("=" * 65)
    print("🎯 FETALYTICS HELD-OUT TEST EVALUATION REPORT")
    print("=" * 65)
    print(f"Model Version Tag:      {rep['version_tag']}")
    print(f"Best Val Epoch:         Epoch {rep.get('best_epoch', 'N/A')}")
    print(f"Held-out Test Dice:     {seg['fetal_head_dice_mean']} (95% CI: {seg['fetal_head_dice_ci95']})")
    print(f"Held-out Test IoU:      {seg['fetal_head_iou_mean']}")
    print(f"Baseline Otsu Dice:     {seg['baseline_otsu_fetal_head_dice_mean']}")
    print(f"Shallow Baseline Dice:  {seg['baseline_supervised_shallow_fetal_head_dice_mean']}")
    print(f"Pubic Symphysis Dice:   {seg['pubic_symphysis_dice_mean']} (95% CI: {seg['pubic_symphysis_dice_ci95']})")
    print(f"Standard Plane F1:      {cls_m['f1_score']} (Sens: {cls_m['sensitivity']}, Spec: {cls_m['specificity']})")
    print(f"QC Accepted Contours:   {bio['qc_accepted_contours']} / {bio['qc_accepted_contours'] + bio['qc_rejected_contours']}")
    print(f"Backblaze B2 Key:       {rep.get('b2_artifact_key')}")
    print("=" * 65)

# Display held-out test diagnostic comparison sheets (Original, Ground Truth, v1.0, v2.0, Overlay)
for sheet_p in sorted(Path("/content/Fetalytics/results/diagnostics").glob("heldout_test_comparison_sheet_*.png")):
    print(f"\nDiagnostic Image: {sheet_p.name}")
    display(Image(filename=str(sheet_p), width=900))